# Gabor Filter Bank

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
from numpy.typing import NDArray
import matplotlib.pyplot as plt
from PIL import Image
import ipywidgets as widgets
from IPython.display import display

In [ ]:
SAMPLE_IMG_DIR = next(
    (path for path in (Path.cwd() / "imgs", Path.cwd().parent / "imgs") if path.is_dir()),
    None,
)
if SAMPLE_IMG_DIR is None:
    raise FileNotFoundError("Could not locate the repository's imgs directory")

image_paths = sorted(
    p for p in SAMPLE_IMG_DIR.iterdir()
    if p.suffix.lower() in {".png", ".jpg", ".jpeg"}
)

image_picker = widgets.Dropdown(
    options=[(p.name, p) for p in image_paths],
    description="Image: ",
    layout=widgets.Layout(width="500px"),
)

display(image_picker)

In [ ]:
GrayF = NDArray[np.float64]  # HxW in [0,1]

def load_grayscale(path: str | Path) -> GrayF:
    """Load an image as grayscale float64 in [0,1]."""
    img = Image.open(path).convert("L")
    return np.asarray(img, dtype=np.float64) / 255.0

## Gabor Kernel

A 2-D Gabor filter is the product of a Gaussian envelope and a sinusoidal carrier.

| Parameter | Meaning |
|---|---|
| $\theta$ | Orientation |
| $\lambda$ | Wavelength (pixels per cycle) |
| $\gamma$ | Spatial aspect ratio (ellipticity) |
| $\sigma$ | Gaussian envelope width |
| $\psi$ | Phase offset |

In [ ]:
## Parameters
thetas: list[float] = list(np.linspace(0, np.pi, 6, endpoint=False))  # 0°, 30°, 60°, 90°, 120°, 150°
lambdas: list[float] = [6.0, 12.0, 24.0]  # pixels per cycle (small -> fine; large -> coarse)
gamma: float = 0.5  # spatial aspect ratio (ellipticity)
# sigma tied to lambda for ~1 octave bandwidth (rule of thumb); sigma ≈ 0.56*lambda
sigma_factor: float = 0.56

In [ ]:
def gabor_kernel(
    size: int,
    sigma: float,
    lam: float,
    theta: float,
    psi: float,
    gamma: float
) -> GrayF:
    """Generate a 2D Gabor filter."""
    r = size // 2
    y, x = np.mgrid[-r:r+1, -r:r+1]

    # rotate coords
    xr = x*np.cos(theta) + y*np.sin(theta)
    yr = -x*np.sin(theta) + y*np.cos(theta)
    gauss = np.exp(-(xr**2 + (gamma**2) * (yr**2)) / (2 * sigma**2))
    carrier = np.cos(2*np.pi*xr/lam + psi)
    g = gauss * carrier
    g -= g.mean()

    # normalize L2 to maintain comparable energy across sizes
    norm = np.linalg.norm(g.ravel()) + 1e-12
    return g / norm

## FFT Convolution

Convolution in the spatial domain equals pointwise multiplication in the frequency domain.

$$
(f * g)(x, y) = \mathcal{F}^{-1}\!\bigl[\mathcal{F}[f] \cdot \mathcal{F}[g]\bigr]
$$
 
The kernel is flipped to perform true *convolution*, then zero-padded to avoid wrap-around artifacts before extracting the central "same"-sized region.


In [ ]:
def fft_convolve2d(image: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    H, W = image.shape
    Kh, Kw = kernel.shape
    # full convolution via FFT
    outH, outW = H + Kh - 1, W + Kw - 1
    f_img = np.fft.rfft2(image, s=(outH, outW))
    # flip kernel for convolution (vs correlation)
    k_flip = np.flipud(np.fliplr(kernel))
    f_ker = np.fft.rfft2(k_flip, s=(outH, outW))
    conv_full = np.fft.irfft2(f_img * f_ker, s=(outH, outW))
    # extract the central region to match image size
    y0 = (Kh - 1) // 2
    x0 = (Kw - 1) // 2
    conv_same = conv_full[y0:y0+H, x0:x0+W]
    return conv_same


def normalize01(x: np.ndarray) -> np.ndarray:
    x = x - x.min()
    d = x.max() - x.min()
    return x / (d + 1e-12)

## Quadrature Energy Envelope

A single Gabor filter is phase-sensitive — its response depends on where the carrier's peaks align with image edges. To obtain a **phase-invariant energy measure**, convolve with a quadrature pair ($\psi = 0$ and $\psi = \pi/2$) and take the amplitude:

$$
E(x,y) = \sqrt{r_{\text{even}}^2(x,y) + r_{\text{odd}}^2(x,y)}
$$

In [ ]:
# build image
img = load_grayscale(image_picker.value)

# compute responses
responses: dict[tuple[float, float], np.ndarray] = {}
energies: dict[tuple[float, float], float] = {}

for lam in lambdas:
    sigma = sigma_factor * lam
    # kernel size ~ 8*sigma (odd)
    ksize = int(np.ceil(8.0 * sigma))
    if ksize % 2 == 0:
        ksize += 1

    for th in thetas:
        # Quadrature pair: psi=0 (even), psi=pi/2 (odd) -> amplitude envelope
        g_even = gabor_kernel(ksize, sigma, lam, th, psi=0.0, gamma=gamma)
        g_odd = gabor_kernel(ksize, sigma, lam, th, psi=np.pi/2, gamma=gamma)
        r_even = fft_convolve2d(img, g_even)
        r_odd = fft_convolve2d(img, g_odd)
        amp = np.sqrt(r_even**2 + r_odd**2)  # phase-insensitive energy
        responses[(lam, th)] = amp
        energies[(lam, th)] = float(np.sum(amp))

In [ ]:
# build bank mosaic image (3x6 grid)
tile_h, tile_w = img.shape
GAP = 2  # gutter between tiles
rows = len(lambdas)
cols = len(thetas)
mosaic_h = rows * tile_h + (rows - 1) * GAP
mosaic_w = cols * tile_w + (cols - 1) * GAP
mosaic = np.zeros((mosaic_h, mosaic_w), dtype=np.float64)  # black background

for i, lam in enumerate(lambdas):
    for j, th in enumerate(thetas):
        tile = normalize01(responses[(lam, th)])
        y0 = i*(tile_h + GAP)
        x0 = j*(tile_w + GAP)
        mosaic[y0:y0+tile_h, x0:x0+tile_w] = tile

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(mosaic, cmap="gray")
ax.set_title("Gabor filter bank responses")
ax.axis("off")
plt.tight_layout()
plt.show()